# Notebook 03: Constraint Activity Analysis

### Project
NEM Network Constraints & Price Divergence Intelligence System

### Business Question
How do network constraints contribute to price spikes, congestion signals, and regional price behaviour in the NEM?

### Purpose Of This Notebook
Notebook 03 introduces the first network layer of the project: dispatch constraint activity.

In Notebook 01, we extracted and cleaned NSW1 and VIC1 price, demand, generation, and net interchange data.

In Notebook 02, we created base market features such as supply-demand gap, price spike flags, price changes, rolling volatility, and time features.

Those features tell us **when the market looked stressed**, but they do not explain whether network limitations were involved.

This notebook begins that explanation by analysing constraint activity from `raw.dispatch_constraints`.


## How This Notebook Links To The Full Project

The full project is designed to explain whether price outcomes are driven by normal market fundamentals or by network conditions.

The project flow is:

1. Build clean regional price and demand data.
2. Create base market features.
3. Analyse constraint activity.
4. Analyse interconnector flows.
5. Measure NSW1 vs VIC1 price divergence.
6. Classify price spikes by likely driver.
7. Translate findings into decision intelligence.

Notebook 03 supports steps 3 and part of step 6.

The output from this notebook will later be joined to the base market feature table so that spike intervals can be tested against constraint activity.


## Key Market Concepts

### Dispatch constraint
A dispatch constraint is a mathematical limit used by AEMO's dispatch engine to keep the power system secure. Constraints can reflect thermal limits, voltage limits, stability limits, outage conditions, or other network/security requirements.

### Marginal value
Constraint marginal value indicates the price impact of relaxing a binding constraint by one unit. A high marginal value can suggest that the constraint is materially affecting dispatch and price outcomes.

### Violation degree
Violation degree indicates whether the dispatch solution is violating the constraint. Any material violation is a risk signal and should be reviewed carefully.

### Active constraint
For this analysis, a constraint is treated as active when it has a non-zero marginal value or a non-zero violation degree.

This is a practical analytical definition for portfolio work. Later, if constraint RHS and equation data are used, the definition can be refined.


## Analyst Objective

By the end of this notebook, we want to create interval-level constraint features:

- active_constraint_count
- max_constraint_marginal_value
- total_constraint_marginal_value
- violation_flag
- max_violation_degree

These features will help answer:

- Were high-price intervals also high-constraint-activity intervals?
- Did constraint marginal values rise during price spikes?
- Were there violation signals during stressed market periods?
- Which intervals should be investigated further in spike classification?


## Inputs And Outputs

### Inputs
This notebook uses:

- `outputs/02_base_market_features.csv`
- `raw.dispatch_constraints`

### Outputs
This notebook will create:

- `outputs/03_constraint_features.csv`
- `outputs/03_constraint_event_summary.csv`
- `outputs/03_market_features_with_constraints.csv`

The final output joins constraint features to the base market feature table for NSW1 and VIC1.


## Step 1: Import Libraries

### What we are doing
We are importing the Python libraries needed for database extraction, data manipulation, and environment variable loading.

### Why it matters
This notebook needs both local CSV data from Notebook 02 and fresh SQL extraction from PostgreSQL.


In [1]:
from pathlib import Path
import os

import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 100)


## Step 2: Project Paths And Analysis Settings



In [2]:
PROJECT_ROOT = (
    Path.cwd().parents[0]
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"

BASE_FEATURES_FILE = OUTPUT_DIR / "02_base_market_features.csv"

START_DATE = "2026-02-01"
END_DATE = "2026-03-01"
REGIONS = ["NSW1", "VIC1"]

print("Project root:", PROJECT_ROOT)
print("Input base features:", BASE_FEATURES_FILE)
print("Output directory:", OUTPUT_DIR)
print("Date range:", START_DATE, "to", END_DATE)
print("Regions:", REGIONS)


Project root: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence
Input base features: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/02_base_market_features.csv
Output directory: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs
Date range: 2026-02-01 to 2026-03-01
Regions: ['NSW1', 'VIC1']


In [3]:
load_dotenv(PROJECT_ROOT / ".env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Database engine created.")


Database engine created.


In [4]:
with engine.connect() as conn:
    result = conn.execute(text("""
        select
            current_database() as database_name,
            current_user as user_name
    """))
    for row in result:
        print(row)


('nemweb', 'vivekarya')


## Step 5: Load Base Market Features From Notebook 02

### What we are doing
We are loading the engineered base market features created in Notebook 02.

### Why it matters
Constraint activity will later be joined to this table at the dispatch interval level. This lets us compare price spikes and market volatility against network constraint activity.


In [6]:
market_features = pd.read_csv(
    BASE_FEATURES_FILE,
    parse_dates=["settlementdate"]
)

print("Market features shape:", market_features.shape)
market_features.head()


Market features shape: (16126, 16)


,settlementdate,regionid,rrp,intervention,totaldemand,availablegeneration,netinterchange,intervention_regionsum,supply_demand_gap,price_spike_flag,price_change,rolling_rrp_volatility_1h,date,hour,weekday,is_weekend
0,2026-02-01 00:05:00,NSW1,57.05984,0,8186.33,13272.53373,-1054.73,0,5086.20373,False,NaN,NaN,2026-02-01,0,Sunday,True
1,2026-02-01 00:10:00,NSW1,64.51979,0,8165.29,13246.75420,-924.67,0,5081.46420,False,7.45995,NaN,2026-02-01,0,Sunday,True
2,2026-02-01 00:15:00,NSW1,65.08727,0,8202.70,13198.51267,-1006.50,0,4995.81267,False,0.56748,4.479816,2026-02-01,0,Sunday,True
3,2026-02-01 00:20:00,NSW1,64.89000,0,8213.41,13169.71135,-1011.73,0,4956.30135,False,-0.19727,3.893369,2026-02-01,0,Sunday,True
4,2026-02-01 00:25:00,NSW1,63.47180,0,8055.53,13139.84760,-934.04,0,5084.31760,False,-1.41820,3.381808,2026-02-01,0,Sunday,True


## Step 6: Check Dispatch Constraint Table Columns

### What we are doing
We are inspecting the available columns in `raw.dispatch_constraints`.

### Why it matters
AEMO ETL pipelines can use slightly different column naming. Before writing the extraction query, we need to confirm the exact fields available for constraint marginal value and violation degree.


In [7]:
constraint_columns_query = text("""
select
    column_name,
    data_type
from information_schema.columns
where table_schema = 'raw'
  and table_name = 'dispatch_constraints'
order by ordinal_position;
""")

constraint_columns_df = pd.read_sql(constraint_columns_query, engine)

constraint_columns_df


,column_name,data_type
0,settlementdate,timestamp without time zone
1,constraintid,text
2,rhs,numeric
3,marginalvalue,numeric
4,lastchanged,text
5,violationdegree,bigint


## Step 7: Check Dispatch Constraint Row Count

### What we are doing
We are checking how many constraint records exist in PostgreSQL for the February 2026 study period.

### Why it matters
Constraint data is much larger than regional price data because there can be many constraints for each dispatch interval. A high row count is normal.


In [8]:
constraint_row_count_query = text("""
select
    count(*) as row_count
from raw.dispatch_constraints
where settlementdate >= :start_date
  and settlementdate < :end_date;
""")

constraint_row_count_df = pd.read_sql(
    constraint_row_count_query,
    engine,
    params={
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
)

constraint_row_count_df


,row_count
0,7386012


## Step 8: Extract Dispatch Constraint Data

### What we are doing
We are extracting dispatch constraint records for the February 2026 study period.

### Why it matters
This is the raw network constraint layer. Each interval can contain many constraint records, so we will later aggregate them into interval-level features that can be joined to regional price data.


In [9]:
constraints_query = text("""
select
    settlementdate,
    constraintid,
    marginalvalue,
    violationdegree
from raw.dispatch_constraints
where settlementdate >= :start_date
  and settlementdate < :end_date
order by settlementdate, constraintid;
""")

constraints_df = pd.read_sql(
    constraints_query,
    engine,
    params={
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
)

print("Constraints shape:", constraints_df.shape)
constraints_df.head()


Constraints shape: (7386012, 4)


,settlementdate,constraintid,marginalvalue,violationdegree
0,2026-02-01 00:05:00,D_I+BIP_ML2_L1,0.0,0
1,2026-02-01 00:05:00,D_I+NIL_MG2_R1,0.0,0
2,2026-02-01 00:05:00,D_NIL_SNOW_N+S_100,0.0,0
3,2026-02-01 00:05:00,D_Q+BIP_ML_L1,0.0,0
4,2026-02-01 00:05:00,D_Q+BIP_ML_L6,0.0,0


## Step 9: Clean Dispatch Constraint Data

### What we are doing
We are standardising timestamps, constraint IDs, and numeric fields.

### Why it matters
Clean constraint data is required before calculating active constraint counts, marginal value intensity, and violation flags.


In [10]:
constraints_clean = constraints_df.copy()

constraints_clean.columns = constraints_clean.columns.str.lower()

constraints_clean["settlementdate"] = pd.to_datetime(
    constraints_clean["settlementdate"]
)

constraints_clean["constraintid"] = (
    constraints_clean["constraintid"]
    .astype(str)
    .str.upper()
    .str.strip()
)

constraints_clean["marginalvalue"] = pd.to_numeric(
    constraints_clean["marginalvalue"],
    errors="coerce"
).fillna(0)

constraints_clean["violationdegree"] = pd.to_numeric(
    constraints_clean["violationdegree"],
    errors="coerce"
).fillna(0)

constraints_clean.head()


,settlementdate,constraintid,marginalvalue,violationdegree
0,2026-02-01 00:05:00,D_I+BIP_ML2_L1,0.0,0
1,2026-02-01 00:05:00,D_I+NIL_MG2_R1,0.0,0
2,2026-02-01 00:05:00,D_NIL_SNOW_N+S_100,0.0,0
3,2026-02-01 00:05:00,D_Q+BIP_ML_L1,0.0,0
4,2026-02-01 00:05:00,D_Q+BIP_ML_L6,0.0,0


## Step 10: Create Active Constraint Flag

### What we are doing
We are flagging constraints as active when they have either a non-zero marginal value or a non-zero violation degree.

### Why it matters
This practical definition identifies constraints that appear to be influencing dispatch outcomes or showing violation risk during an interval.


In [11]:
constraints_clean["active_constraint_flag"] = (
    (constraints_clean["marginalvalue"].abs() > 0)
    | (constraints_clean["violationdegree"].abs() > 0)
)

constraints_clean[
    [
        "settlementdate",
        "constraintid",
        "marginalvalue",
        "violationdegree",
        "active_constraint_flag",
    ]
].head()


,settlementdate,constraintid,marginalvalue,violationdegree,active_constraint_flag
0,2026-02-01 00:05:00,D_I+BIP_ML2_L1,0.0,0,False
1,2026-02-01 00:05:00,D_I+NIL_MG2_R1,0.0,0,False
2,2026-02-01 00:05:00,D_NIL_SNOW_N+S_100,0.0,0,False
3,2026-02-01 00:05:00,D_Q+BIP_ML_L1,0.0,0,False
4,2026-02-01 00:05:00,D_Q+BIP_ML_L6,0.0,0,False


## Step 11: Aggregate Constraints To Dispatch Interval Level

### What we are doing
We are converting many constraint rows per dispatch interval into one constraint feature row per interval.

### Why it matters
The market feature table has one row per region per dispatch interval. Constraint data has many rows per dispatch interval. To join constraints to market prices, we first need interval-level constraint features.

### Features created
- active_constraint_count
- max_constraint_marginal_value
- total_constraint_marginal_value
- violation_flag
- max_violation_degree


In [12]:
constraint_features = (
    constraints_clean
    .groupby("settlementdate")
    .agg(
        active_constraint_count=("active_constraint_flag", "sum"),
        max_constraint_marginal_value=("marginalvalue", "max"),
        total_constraint_marginal_value=("marginalvalue", "sum"),
        violation_flag=("violationdegree", lambda x: (x.abs() > 0).any()),
        max_violation_degree=("violationdegree", "max"),
    )
    .reset_index()
)

constraint_features.head()


,settlementdate,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree
0,2026-02-01 00:05:00,17,135.3015,-902.14975,False,0
1,2026-02-01 00:10:00,16,40.2794,-1006.87256,False,0
2,2026-02-01 00:15:00,20,9.1600,-1053.75439,False,0
3,2026-02-01 00:20:00,20,4.3600,-1061.42631,False,0
4,2026-02-01 00:25:00,20,4.3600,-1060.65174,False,0


## Step 12: Validate Constraint Feature Coverage

### What we are doing
We are checking the number of dispatch intervals covered by the constraint feature table.

### Why it matters
Constraint features should have one row for each dispatch interval in the study period. This allows a clean join to NSW1 and VIC1 market features.


In [13]:
constraint_feature_coverage = pd.DataFrame({
    "first_interval": [constraint_features["settlementdate"].min()],
    "last_interval": [constraint_features["settlementdate"].max()],
    "interval_count": [constraint_features["settlementdate"].nunique()],
    "max_active_constraints": [constraint_features["active_constraint_count"].max()],
    "max_marginal_value": [constraint_features["max_constraint_marginal_value"].max()],
    "max_violation_degree": [constraint_features["max_violation_degree"].max()],
})

constraint_feature_coverage


,first_interval,last_interval,interval_count,max_active_constraints,max_marginal_value,max_violation_degree
0,2026-02-01 00:05:00,2026-02-28 23:55:00,8063,33,7.510919e+06,123


## Step 13: Identify High Constraint Activity Intervals

### What we are doing
We are listing intervals with the highest active constraint counts and marginal values.

### Why it matters
These intervals are likely candidates for network-driven market behaviour and will be compared against price spikes and price divergence later.


In [14]:
high_constraint_activity = (
    constraint_features
    .sort_values(
        ["active_constraint_count", "max_constraint_marginal_value"],
        ascending=False
    )
    .head(20)
)

high_constraint_activity


,settlementdate,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree
1594,2026-02-06 12:55:00,33,3.98000,-9799.73615,False,0
5603,2026-02-20 11:00:00,31,55.28000,-10314.67060,False,0
5622,2026-02-20 12:35:00,31,52.78000,-11792.99542,False,0
1036,2026-02-04 14:25:00,31,7.80000,-11374.48678,False,0
2758,2026-02-10 13:55:00,31,7.80000,-6424.80494,False,0
5609,2026-02-20 11:30:00,31,7.59622,-11461.21517,False,0
2759,2026-02-10 14:00:00,31,4.96000,-7583.95214,False,0
996,2026-02-04 11:05:00,30,162400.00000,152346.38932,True,0
1322,2026-02-05 14:15:00,30,124.03234,-5125.92495,False,0
5606,2026-02-20 11:15:00,30,55.28000,-10347.19801,False,0


## Step 14: Join Constraint Features To Market Features

### What we are doing
We are joining interval-level constraint features to the NSW1 and VIC1 market feature table.

### Why it matters
This creates a combined table where each regional price interval can be compared with system-wide constraint activity at the same dispatch interval.


In [15]:
market_features_with_constraints = market_features.merge(
    constraint_features,
    on="settlementdate",
    how="left"
)

constraint_feature_cols = [
    "active_constraint_count",
    "max_constraint_marginal_value",
    "total_constraint_marginal_value",
    "violation_flag",
    "max_violation_degree",
]

market_features_with_constraints[constraint_feature_cols] = (
    market_features_with_constraints[constraint_feature_cols]
    .fillna({
        "active_constraint_count": 0,
        "max_constraint_marginal_value": 0,
        "total_constraint_marginal_value": 0,
        "violation_flag": False,
        "max_violation_degree": 0,
    })
)

print("Before join:", market_features.shape)
print("After join:", market_features_with_constraints.shape)

market_features_with_constraints.head()


Before join: (16126, 16)
After join: (16126, 21)


,settlementdate,regionid,rrp,intervention,totaldemand,availablegeneration,netinterchange,intervention_regionsum,supply_demand_gap,price_spike_flag,price_change,rolling_rrp_volatility_1h,date,hour,weekday,is_weekend,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree
0,2026-02-01 00:05:00,NSW1,57.05984,0,8186.33,13272.53373,-1054.73,0,5086.20373,False,NaN,NaN,2026-02-01,0,Sunday,True,17,135.3015,-902.14975,False,0
1,2026-02-01 00:10:00,NSW1,64.51979,0,8165.29,13246.75420,-924.67,0,5081.46420,False,7.45995,NaN,2026-02-01,0,Sunday,True,16,40.2794,-1006.87256,False,0
2,2026-02-01 00:15:00,NSW1,65.08727,0,8202.70,13198.51267,-1006.50,0,4995.81267,False,0.56748,4.479816,2026-02-01,0,Sunday,True,20,9.1600,-1053.75439,False,0
3,2026-02-01 00:20:00,NSW1,64.89000,0,8213.41,13169.71135,-1011.73,0,4956.30135,False,-0.19727,3.893369,2026-02-01,0,Sunday,True,20,4.3600,-1061.42631,False,0
4,2026-02-01 00:25:00,NSW1,63.47180,0,8055.53,13139.84760,-934.04,0,5084.31760,False,-1.41820,3.381808,2026-02-01,0,Sunday,True,20,4.3600,-1060.65174,False,0


## Step 15: Compare Constraint Activity During Spike And Non-Spike Intervals

### What we are doing
We are comparing average constraint activity between price spike intervals and normal intervals.

### Why it matters
If spike intervals show higher active constraint counts or higher marginal values, that is an early signal that network constraints may be contributing to price outcomes.


In [16]:
constraint_vs_spike_summary = (
    market_features_with_constraints
    .groupby(["regionid", "price_spike_flag"])
    .agg(
        intervals=("settlementdate", "count"),
        average_rrp=("rrp", "mean"),
        max_rrp=("rrp", "max"),
        average_active_constraints=("active_constraint_count", "mean"),
        max_active_constraints=("active_constraint_count", "max"),
        average_max_constraint_marginal_value=("max_constraint_marginal_value", "mean"),
        max_constraint_marginal_value=("max_constraint_marginal_value", "max"),
        violation_intervals=("violation_flag", "sum"),
    )
    .reset_index()
)

constraint_vs_spike_summary


,regionid,price_spike_flag,intervals,average_rrp,max_rrp,average_active_constraints,max_active_constraints,average_max_constraint_marginal_value,max_constraint_marginal_value,violation_intervals
0,NSW1,False,8010,68.323133,299.82000,19.259051,33,82322.258384,7.510919e+06,54
1,NSW1,True,53,2417.098468,20300.00000,24.415094,30,9.248682,3.783458e+01,0
2,VIC1,False,8063,45.055455,269.77601,19.292943,33,81781.195564,7.510919e+06,54


## Interpretation: Constraint Activity During Spike And Non-Spike Intervals

The comparison between spike and non-spike intervals shows that NSW1 recorded 53 price spike intervals above $300/MWh during the February 2026 study period, while VIC1 did not record any spike intervals under this threshold.

For NSW1, average RRP increased sharply from about $68/MWh during non-spike intervals to about $2,417/MWh during spike intervals. At the same time, the average number of active constraints increased from around 19.3 to 24.4. This indicates that NSW1 spike intervals occurred during periods of higher constraint activity.

However, the maximum constraint marginal value signal does not directly support a simple constraint-driven explanation. NSW1 non-spike intervals had a much higher average maximum constraint marginal value than spike intervals. The largest marginal value events also occurred outside the spike intervals. In addition, there were no constraint violation intervals during NSW1 spike periods.

This means the NSW1 price spikes should not be classified as purely constraint-driven based on this evidence alone. Constraint activity may have contributed to tighter or more complex dispatch conditions, but the marginal value and violation signals do not show a direct spike-causing constraint effect.

The next analytical step is to examine interconnector flows and regional price divergence. If NSW1 prices separated from VIC1 during these spike intervals, and if interconnector flows were constrained or changing sharply, the spike explanation may be more consistent with interconnector or congestion-driven behaviour rather than constraint marginal value alone.


## Step 16: Create Constraint Event Summary

### What we are doing
We are creating a summary of the highest constraint activity intervals.

### Why it matters
This table helps identify dispatch intervals where network constraint activity was strongest. These intervals can be reviewed later against price spikes, interconnector flows, and price divergence.


In [17]:
constraint_event_summary = (
    constraint_features
    .sort_values(
        ["active_constraint_count", "max_constraint_marginal_value"],
        ascending=False
    )
    .head(100)
    .reset_index(drop=True)
)

constraint_event_summary.head(20)


,settlementdate,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree
0,2026-02-06 12:55:00,33,3.98000,-9799.73615,False,0
1,2026-02-20 11:00:00,31,55.28000,-10314.67060,False,0
2,2026-02-20 12:35:00,31,52.78000,-11792.99542,False,0
3,2026-02-04 14:25:00,31,7.80000,-11374.48678,False,0
4,2026-02-10 13:55:00,31,7.80000,-6424.80494,False,0
5,2026-02-20 11:30:00,31,7.59622,-11461.21517,False,0
6,2026-02-10 14:00:00,31,4.96000,-7583.95214,False,0
7,2026-02-04 11:05:00,30,162400.00000,152346.38932,True,0
8,2026-02-05 14:15:00,30,124.03234,-5125.92495,False,0
9,2026-02-20 11:15:00,30,55.28000,-10347.19801,False,0


## Step 17: Create Spike Constraint Detail Table

### What we are doing
We are filtering the joined market and constraint table to show only price spike intervals.

### Why it matters
This table focuses on the high-price events that will later be classified by likely driver. It shows whether those intervals had elevated constraint counts, high marginal values, or violation signals.


In [18]:
spike_constraint_detail = (
    market_features_with_constraints[
        market_features_with_constraints["price_spike_flag"]
    ]
    .sort_values(["rrp"], ascending=False)
    .reset_index(drop=True)
)

spike_constraint_detail[
    [
        "settlementdate",
        "regionid",
        "rrp",
        "totaldemand",
        "availablegeneration",
        "supply_demand_gap",
        "active_constraint_count",
        "max_constraint_marginal_value",
        "total_constraint_marginal_value",
        "violation_flag",
        "max_violation_degree",
    ]
].head(20)


,settlementdate,regionid,rrp,totaldemand,availablegeneration,supply_demand_gap,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree
0,2026-02-05 14:30:00,NSW1,20300.00000,10190.15,15492.52355,5302.37355,24,12.12000,-100159.89494,False,0
1,2026-02-06 14:05:00,NSW1,19036.53000,9651.93,15075.06206,5423.13206,23,5.34000,-61802.29740,False,0
2,2026-02-05 18:05:00,NSW1,9914.02372,11811.76,13709.77609,1898.01609,22,20.23000,-22761.00366,False,0
3,2026-02-05 17:55:00,NSW1,9911.42681,11901.45,14019.02209,2117.57209,24,4.96000,-22639.48874,False,0
4,2026-02-05 16:35:00,NSW1,9240.83000,11896.59,14384.92803,2488.33803,29,4.34000,-29431.52628,False,0
5,2026-02-07 16:40:00,NSW1,9199.30000,11073.40,13443.29640,2369.89640,21,10.41000,-20634.31200,False,0
6,2026-02-05 14:50:00,NSW1,9199.30000,10400.88,15793.28831,5392.40831,23,7.45000,-32788.78141,False,0
7,2026-02-06 13:35:00,NSW1,9199.30000,9613.17,15758.70193,6145.53193,26,7.99000,-60699.49111,False,0
8,2026-02-05 14:40:00,NSW1,7380.80815,10203.66,15819.78989,5616.12989,24,9.19000,-22251.99604,False,0
9,2026-02-05 18:15:00,NSW1,1278.88000,11744.07,13349.01575,1604.94575,21,3.96000,-6279.40831,False,0


## Step 18: Save Notebook 03 Outputs

### What we are doing
We are saving the constraint feature table, high-activity event summary, spike constraint detail table, and market features joined with constraint features.

### Why it matters
These outputs become inputs for later notebooks. In particular, `03_market_features_with_constraints.csv` will be used when we classify spike events after adding interconnector flow features.


In [19]:
constraint_features_output = OUTPUT_DIR / "03_constraint_features.csv"
constraint_event_summary_output = OUTPUT_DIR / "03_constraint_event_summary.csv"
spike_constraint_detail_output = OUTPUT_DIR / "03_spike_constraint_detail.csv"
market_with_constraints_output = OUTPUT_DIR / "03_market_features_with_constraints.csv"

constraint_features.to_csv(constraint_features_output, index=False)
constraint_event_summary.to_csv(constraint_event_summary_output, index=False)
spike_constraint_detail.to_csv(spike_constraint_detail_output, index=False)
market_features_with_constraints.to_csv(market_with_constraints_output, index=False)

print("Saved:", constraint_features_output)
print("Saved:", constraint_event_summary_output)
print("Saved:", spike_constraint_detail_output)
print("Saved:", market_with_constraints_output)


Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/03_constraint_features.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/03_constraint_event_summary.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/03_spike_constraint_detail.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/03_market_features_with_constraints.csv


## Notebook 03 Analyst Note

Notebook 03 added the first network layer to the Project 2 analysis by extracting and aggregating dispatch constraint activity from `raw.dispatch_constraints`.

The analysis created interval-level constraint features including active constraint count, maximum constraint marginal value, total constraint marginal value, violation flag, and maximum violation degree. These features were joined to the NSW1 and VIC1 base market feature table.

Initial results show that NSW1 price spike intervals occurred during periods with a higher average number of active constraints. However, the marginal value and violation signals do not support a simple conclusion that the NSW1 spikes were purely constraint-driven. The largest marginal value events occurred outside the spike intervals, and no violation intervals were observed during NSW1 spike periods.

This suggests that constraints may have contributed to tighter dispatch conditions, but further analysis is required. The next notebook will examine interconnector flows to test whether NSW1 price spikes and NSW1-VIC1 price separation were associated with flow changes, import/export conditions, or congestion-style behaviour.
